# 15. Value of in-progress detection: how much of the decline is avoided by acting at the alert

In notebook 12 I found that warning **before** the onset did not replicate, while **in-progress detection**, catching a decline already under way at least 30 minutes before the crash, did. Here I measure how useful that detection is in practice, using the existing alerts unchanged (no new policy or threshold).

For the action I follow the execution rule of notebook 04. I decide at the close of the alert minute, sell the BTC position at the close of **the first minute with volume within 5 minutes, starting from the next minute**, and buy it back on the same price basis 240 minutes later (sensitivity: 60 and 1,440 minutes). Avoided loss = −(return from holding the position).

I ask three questions.
1. Event level: for caught events, I measure how far the price had already fallen at the time of action and how much of the further fall to the crash (4%) and to the trough (lowest price within 4 or 24 hours) the action avoided, separately for pre-onset warning and in-progress detection.
2. Alert level: I compute the profit and loss of all alerts, false alerts included. Each alert falls into one class: pre-onset hit, in-progress detection hit, within 30 minutes of the crash, within 24 hours after the crash, too early, or no event nearby. I then compute net value at one-way fees of 0, 5, 10 and 25 bp.
3. Value of timing: I test whether the loss avoided per alert exceeds that of the same alerts moved to random times (circular shift) or to random times in the same volatility regime (regime-matched).

My primary policy is calendar alerts (20 per month, 240 per year), and my secondary one is the constant token bucket (the policy with many in-progress detections). I evaluate both in the development period (2023-01 to 2026-04) and the holdout (2018–2020).

This analysis is exploratory and descriptive. Trough prices are known only in hindsight, so I use them only to show an *upper bound on value*. I record the settings in `VALUE_SPEC.json` and write the results to `detection_value_v1`. Only some cells of this saved copy show their output; `VALUE_SUMMARY.json` and the tables in that folder hold all results.

In [2]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df


OUT = SPOT_BASE / 'detection_value_v1'; OUT.mkdir(parents=True, exist_ok=True)
HOLD = SPOT_BASE / 'HOLDOUT_2018_2020_confirmatory_v1'
RAW_HOLD = DRIVE / 'BTCUSDT_spot_holdout_2017_2020' / '1m'
CAS = SPOT_BASE / 'causal_alert_selection_v1'; RA = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1'
H_START, H_END = '2018-01-01', '2021-01-01'
PERIODS = {'dev': (EVAL_START, EVAL_END), 'holdout': (H_START, H_END)}
ALARM_FILES = {'calendar': {'dev': CAS / 'alarms_first_eligible.csv', 'holdout': HOLD / 'alarms_calendar.csv'},
               'constant_bucket': {'dev': RA / 'alarms_constant_bucket_recalibrated.csv', 'holdout': HOLD / 'alarms_constant_bucket.csv'}}
for _p in [HOLD / 'holdout_events.csv', HOLD / 'spot_download_manifest.csv', HOLD / 'HOLDOUT_RESULTS.json', ALARM_FILES['calendar']['dev'], ALARM_FILES['calendar']['holdout']]:
    if not _p.exists():
        raise FileNotFoundError(f'Results of earlier notebooks are required: {_p}')
HOLDS = [60, 240, 1440]; PRIMARY_HOLD = 240; FEES_BP = [0, 5, 10, 25]; MAX_WAIT = 5
N_NULL = 5000; SEED = 20261029; REGIME_WINDOW_D = 30; RV_WINDOW = 1440; ALPHA = 0.05
SPEC = {'title': 'Value of in-progress detection (exploratory, descriptive)',
        'action': f'signal at the alert minute close; exit at the close of the first positive-volume minute within {MAX_WAIT} min after it; '
                  f're-enter after H minutes (H in {HOLDS}, primary {PRIMARY_HOLD}); avoided loss = -log return over the hold',
        'fees': f'one-way {FEES_BP} bp, charged on exit and re-entry (full position)',
        'event_metrics': 'drawdown from the event peak at entry; decline avoided to the 4% breach close, to the 4h and 24h post-breach minimum close; '
                         'share of the peak-to-24h-trough drawdown avoided (hindsight upper bound)',
        'alert_classes': ['pre-onset hit', 'post-onset hit', 'late (<30 min before breach)', 'after breach (<=24h)',
                          'too early (4-24h before breach)', 'no event within 24h'],
        'nulls': {'circular': f'all alerts shifted by U[7d, L-7d], {N_NULL} draws', 'regime_matched': f'same 24h-RV decile within +-{REGIME_WINDOW_D}d, {N_NULL} draws'},
        'policies': {'calendar': 'notebook 5 / 11 saved alarms (primary)', 'constant_bucket': 'notebook 6 recalibrated / notebook 11 (secondary, if files exist)'},
        'no_new_policy': True}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True) + '\n').encode()
(OUT / 'VALUE_SPEC.json').write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/detection_value_v1 | spec sha256 51cae38eaec8ef0f


## 0. Data and reproduction of earlier results

In [ ]:
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
px, sg = load_btc_spot()
E_btc = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
EVENTS = {'dev': events_between(E_btc, EVAL_START, EVAL_END)}
hev = pd.read_csv(HOLD / 'holdout_events.csv')
for c_ in TCOLS:
    hev[c_] = pd.to_datetime(hev[c_], utc=True)
EVENTS['holdout'] = hev.sort_values('breach_utc').reset_index(drop=True)

man = pd.read_csv(HOLD / 'spot_download_manifest.csv')
for r in man.itertuples():
    if r.status != 'not_listed' and sha256_file(RAW_HOLD / f'{r.symbol}-1m-{r.month}.zip') != r.sha256:
        raise RuntimeError(f'GUARD FAIL: holdout archive {r.month} SHA-256 differs from the notebook-11 manifest')
Hh = regularize(load_archives(man, RAW_HOLD), end=pd.Timestamp('2020-12-31 23:59', tz='UTC')).loc[:pd.Timestamp('2020-12-31 23:59', tz='UTC')]
eh = events_between(price_only_catalog(Hh.close.to_numpy(float), Hh.volume.to_numpy(float), Hh.index), H_START, H_END)
if len(eh) != len(EVENTS['holdout']) or any(not np.array_equal(pd.to_datetime(eh[c_], utc=True).astype('int64').to_numpy(),
                                                              EVENTS['holdout'][c_].astype('int64').to_numpy()) for c_ in TCOLS):
    raise RuntimeError('GUARD FAIL: holdout prices do not reproduce the notebook-11 events')

MKT = {}
for k, P_ in [('dev', px), ('holdout', Hh)]:
    T = P_.index.asi8; c = P_.close.to_numpy(float); v = P_.volume.to_numpy(float)
    good = np.isfinite(c) & (c > 0) & np.isfinite(v) & (v > 0)
    idx = np.where(good, np.arange(len(T)), len(T))
    nxt = np.minimum.accumulate(idx[::-1])[::-1]
    cf = pd.Series(c).ffill().to_numpy()
    rv = pd.Series(np.log(c)).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
    MKT[k] = dict(T=T, c=c, cf=cf, next_good=nxt, rv=rv)
del Hh


def load_alarms(p):
    return np.sort(pd.to_datetime(pd.read_csv(p).alarm_utc, utc=True).astype('int64').to_numpy())


ALERTS = {}
H1 = json.loads((HOLD / 'HOLDOUT_RESULTS.json').read_text())['H1']
ref = {('calendar', 'dev'): REF['first_eligible'], ('calendar', 'holdout'): [H1['alarms'], H1['any_hits'], H1['pre_onset_hits']]}
rac = pd.read_csv(RA / 'risk_adaptive_policy_comparison.csv') if (RA / 'risk_adaptive_policy_comparison.csv').is_file() else None
sec = pd.read_csv(HOLD / 'secondary_policy_results.csv')
if rac is not None:
    r6 = rac[rac.policy == 'constant_bucket_recalibrated'].iloc[0]
    pc6 = next(c_ for c_ in ['before_1pct', 'pre_onset', 'pre_onset_caught', 'before_1pct_onset'] if c_ in rac.columns)
    ref[('constant_bucket', 'dev')] = [int(r6.alarms), int(r6.caught), int(r6[pc6])]
r11 = sec[(sec.policy == 'constant_bucket') & sec.year.isna()].iloc[0]
ref[('constant_bucket', 'holdout')] = [int(r11.alarms), int(r11.caught), int(r11.before_1pct)]
for pol, files in ALARM_FILES.items():
    for k, p in files.items():
        if not p.is_file() or (pol, k) not in ref:
            print(f'skip {pol} {k}: file or reference missing'); continue
        a = load_alarms(p); st = statistics(EVENTS[k], a)
        if [st['alarms'], st['caught'], st['before_1pct']] != ref[(pol, k)]:
            raise RuntimeError(f'GUARD FAIL: {pol} {k} alarms do not reproduce the saved result ({st} vs {ref[(pol, k)]})')
        ALERTS[(pol, k)] = a
print('PASS: alarms reproduce saved results:', {f'{p}/{k}': len(a) for (p, k), a in ALERTS.items()})

## 1. Event level: loss avoided on caught events

In [ ]:
def entry_idx(k, times):
    m = MKT[k]; i0 = np.searchsorted(m['T'], np.asarray(times, np.int64), 'right')
    i0 = np.minimum(i0, len(m['T']) - 1); ie = m['next_good'][i0]
    ok = (ie < len(m['T'])) & (ie - i0 < MAX_WAIT)
    return np.where(ok, ie, -1)


def fwd_avoided(k, ie, H):
    m = MKT[k]; ok = (ie >= 0) & (ie + H < len(m['T']))
    out = np.full(len(ie), np.nan)
    out[ok] = -np.log(m['cf'][ie[ok] + H] / m['cf'][ie[ok]])
    return out


def classify(ev, a):
    m = match_events(ev, a); cls = np.array(['no event within 24h'] * len(a), dtype=object)
    b = np.sort(ev.breach_utc.astype('int64').to_numpy())
    for j, t in enumerate(a):
        q = np.searchsorted(b, t, 'left'); nxt = b[q] if q < len(b) else None; prv = b[q - 1] if q > 0 else None
        if nxt is not None and nxt - t < LEAD_MIN * NS:
            cls[j] = 'late (<30 min before breach)'
        elif prv is not None and t - prv <= 1440 * NS:
            cls[j] = 'after breach (<=24h)'
        elif nxt is not None and LEAD_MAX * NS < nxt - t <= 1440 * NS:
            cls[j] = 'too early (4-24h before breach)'
        elif nxt is not None and LEAD_MIN * NS <= nxt - t <= LEAD_MAX * NS:
            cls[j] = 'in window, not used'
    pos = {int(x.value): i for i, x in enumerate(pd.to_datetime(a, utc=True))}
    for r in m.itertuples():
        cls[pos[int(r.alarm_utc.value)]] = 'pre-onset hit' if r.pre1pct else 'post-onset hit'
    return cls, m


ev_rows = []; ALERT_TABLE = {}
for (pol, k), a in ALERTS.items():
    ev = EVENTS[k]; mk = MKT[k]
    cls, m = classify(ev, a)
    ie = entry_idx(k, a)
    tab = pd.DataFrame({'alarm_utc': pd.to_datetime(a, utc=True), 'class': cls, 'entry_ok': ie >= 0})
    for H in HOLDS:
        tab[f'avoided_{H}m'] = fwd_avoided(k, ie, H)
    ALERT_TABLE[(pol, k)] = tab
    mm = m.merge(ev[['event_id', 'peak_close', 'breach_close', 'onset_1pct_utc']], on='event_id')
    for r in mm.itertuples():
        j = int(np.searchsorted(a, r.alarm_utc.value)); e_ = ie[j]
        if e_ < 0:
            continue
        ib = int(np.searchsorted(mk['T'], r.breach_utc.value))
        pe = mk['cf'][e_]; tr4 = np.nanmin(mk['c'][ib:ib + LEAD_MAX + 1]); tr24 = np.nanmin(mk['c'][ib:ib + 1441])
        ev_rows.append(dict(policy=pol, period=k, event_id=int(r.event_id), endpoint='pre-onset hit' if r.pre1pct else 'post-onset hit',
                            minutes_before_breach=float(r.lead_min), minutes_after_onset=(r.alarm_utc - r.onset_1pct_utc).total_seconds() / 60,
                            drawdown_at_entry_pct=100 * (1 - pe / r.peak_close),
                            avoided_to_breach_pct=100 * (1 - r.breach_close / pe),
                            avoided_to_trough4h_pct=100 * (1 - tr4 / pe), avoided_to_trough24h_pct=100 * (1 - tr24 / pe),
                            peak_to_trough24h_pct=100 * (1 - tr24 / r.peak_close),
                            share_of_drawdown_avoided=(pe - tr24) / (r.peak_close - tr24) if r.peak_close > tr24 else np.nan,
                            avoided_hold_240m_pct=100 * (1 - mk['cf'][min(e_ + PRIMARY_HOLD, len(mk['cf']) - 1)] / pe)))
EVV = pd.DataFrame(ev_rows); EVV.to_csv(OUT / '1_event_level_value.csv', index=False)
agg = EVV.groupby(['policy', 'period', 'endpoint']).agg(
    events=('event_id', 'size'), median_minutes_before_breach=('minutes_before_breach', 'median'),
    median_drawdown_at_entry_pct=('drawdown_at_entry_pct', 'median'), median_avoided_to_breach_pct=('avoided_to_breach_pct', 'median'),
    median_avoided_to_trough4h_pct=('avoided_to_trough4h_pct', 'median'), median_avoided_to_trough24h_pct=('avoided_to_trough24h_pct', 'median'),
    median_share_of_drawdown_avoided=('share_of_drawdown_avoided', 'median'), median_avoided_hold_240m_pct=('avoided_hold_240m_pct', 'median')).reset_index()
agg.to_csv(OUT / '1_event_level_summary.csv', index=False); display(agg.round(3))

cov = []
for (pol, k) in ALERTS:
    ev = EVENTS[k]; mk = MKT[k]; tot = 0.0
    for r in ev.itertuples():
        ib = int(np.searchsorted(mk['T'], r.breach_utc.value)); tr24 = np.nanmin(mk['c'][ib:ib + 1441])
        tot += max(0.0, 1 - tr24 / r.peak_close)
    sub = EVV[(EVV.policy == pol) & (EVV.period == k)]
    for endpoint in ['pre-onset hit', 'post-onset hit']:
        s_ = sub[sub.endpoint == endpoint]
        avoided = float(((s_.avoided_to_trough24h_pct / 100) * (1 - s_.drawdown_at_entry_pct / 100)).clip(lower=0).sum())
        cov.append(dict(policy=pol, period=k, endpoint=endpoint, events_all=len(ev), hits=len(s_),
                        share_of_all_crash_drawdown_avoided=avoided / tot if tot else np.nan))
COV = pd.DataFrame(cov); COV.to_csv(OUT / '1_coverage_of_total_drawdown.csv', index=False); display(COV.round(4))

## 2. Alert level: profit and loss including false alerts

In [ ]:
rows = []
for (pol, k), tab in ALERT_TABLE.items():
    for cl, g in tab.groupby('class'):
        rows.append(dict(policy=pol, period=k, alert_class=cl, alerts=len(g), share_of_alerts=len(g) / len(tab),
                         **{f'mean_avoided_{H}m_bp': 1e4 * float(g[f'avoided_{H}m'].mean()) for H in HOLDS},
                         median_avoided_240m_bp=1e4 * float(g['avoided_240m'].median()),
                         share_positive_240m=float((g['avoided_240m'] > 0).mean())))
AC = pd.DataFrame(rows); AC.to_csv(OUT / '2_alert_classes.csv', index=False)
display(AC.round(2))

rows = []
for (pol, k), tab in ALERT_TABLE.items():
    years = (ts(PERIODS[k][1]).value - ts(PERIODS[k][0]).value) / (365.25 * 1440 * NS)
    x = tab['avoided_240m'].dropna()
    for subset, mask in [('all alerts', np.ones(len(tab), bool)), ('hits only', tab['class'].str.endswith('hit').to_numpy()),
                         ('post-onset hits only', (tab['class'] == 'post-onset hit').to_numpy()),
                         ('non-hits', ~tab['class'].str.endswith('hit').to_numpy())]:
        y = tab.loc[mask, 'avoided_240m'].dropna()
        for fee in FEES_BP:
            net = y - 2 * fee * 1e-4
            rows.append(dict(policy=pol, period=k, subset=subset, one_way_fee_bp=fee, alerts=len(y),
                             mean_net_per_alert_bp=1e4 * float(net.mean()), total_net_pct=100 * float(net.sum()),
                             total_net_pct_per_year=100 * float(net.sum()) / years))
NV = pd.DataFrame(rows); NV.to_csv(OUT / '2_net_value_by_fee.csv', index=False)
display(NV[NV.subset.isin(['all alerts', 'post-onset hits only'])].pivot_table(index=['policy', 'period', 'subset'], columns='one_way_fee_bp',
                                                                                  values='total_net_pct_per_year').round(2))

,policy,period,alert_class,alerts,share_of_alerts,mean_avoided_60m_bp,mean_avoided_240m_bp,mean_avoided_1440m_bp,median_avoided_240m_bp,share_positive_240m
0,calendar,dev,after breach (<=24h),120,0.15,-8.29,-11.37,-91.26,-22.56,0.40
1,calendar,dev,late (<30 min before breach),11,0.01,225.39,193.32,180.25,175.21,0.91
2,calendar,dev,no event within 24h,577,0.72,-2.58,-12.52,-39.58,-10.08,0.46
3,calendar,dev,post-onset hit,17,0.02,160.97,265.82,256.09,272.06,1.00
4,calendar,dev,pre-onset hit,22,0.03,104.00,338.00,224.12,354.58,1.00
5,calendar,dev,too early (4-24h before breach),53,0.07,-30.66,-18.95,317.10,-25.88,0.34
6,calendar,holdout,after breach (<=24h),168,0.24,7.49,-10.51,2.42,-0.06,0.50
7,calendar,holdout,late (<30 min before breach),18,0.03,149.32,152.47,272.54,128.81,0.78
8,calendar,holdout,no event within 24h,430,0.60,-7.97,-20.06,-61.64,-14.79,0.42
9,calendar,holdout,post-onset hit,23,0.03,116.87,170.07,273.68,149.53,0.83


one_way_fee_bp                                   0      5      10      25
policy          period  subset                                           
calendar        dev     all alerts            13.49 -10.54 -34.57 -106.66
                        post-onset hits only  13.57  13.06  12.55   11.02
                holdout all alerts           -25.51 -49.27 -73.03 -144.32
                        post-onset hits only  13.04  12.27  11.50    9.20
constant_bucket dev     all alerts           -10.87 -34.74 -58.62 -130.26
                        post-onset hits only  22.34  21.44  20.53   17.83
                holdout all alerts           -25.67 -49.13 -72.59 -142.97
                        post-onset hits only  10.62   9.79   8.96    6.46

## 3. Value of timing: loss avoided per alert versus random times

In [ ]:
def deciles(k):
    m = MKT[k]; s, e = PERIODS[k]; ok = in_period(m['T'], s, e) & np.isfinite(m['rv'])
    tt = m['T'][ok]; d = np.minimum((pd.Series(m['rv'][ok]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    return tt, d


DEC = {k: deciles(k) for k in PERIODS}


def regime_sampler(k, a):
    tt, d = DEC[k]
    order = np.lexsort((tt, d)); P = tt[order]; ds = d[order]
    starts = np.searchsorted(ds, np.arange(10), 'left'); ends = np.searchsorted(ds, np.arange(10), 'right')
    j = np.clip(np.searchsorted(tt, a, 'right') - 1, 0, len(tt) - 1); ad = d[j]; W = REGIME_WINDOW_D * 1440 * NS
    lo = np.array([starts[q] + np.searchsorted(P[starts[q]:ends[q]], ti - W, 'left') for ti, q in zip(a, ad)])
    hi = np.array([starts[q] + np.searchsorted(P[starts[q]:ends[q]], ti + W, 'right') for ti, q in zip(a, ad)])

    def draw(rng):
        idx = lo + (rng.random(len(a)) * (hi - lo)).astype(np.int64); c = P[idx]; bad = np.abs(c - a) < LEAD_MAX * NS
        for _ in range(20):
            if not bad.any():
                break
            c[bad] = P[lo[bad] + (rng.random(int(bad.sum())) * (hi[bad] - lo[bad])).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
        return np.sort(c)
    return draw


rows = []; NULLS = {}
t_ = time.time()
for (pol, k), a in ALERTS.items():
    s, e = PERIODS[k]; t0 = ts(s).value; L = (ts(e).value - t0) // NS
    obs = {H: float(np.nanmean(fwd_avoided(k, entry_idx(k, a), H))) for H in HOLDS}
    rng = np.random.default_rng(SEED); deltas = rng.integers(7 * 1440, L - 7 * 1440, size=N_NULL)
    circ = {H: np.empty(N_NULL) for H in HOLDS}; reg = {H: np.empty(N_NULL) for H in HOLDS}
    draw = regime_sampler(k, a); rng2 = np.random.default_rng(SEED + 1)
    for i in range(N_NULL):
        sh = np.sort(t0 + (a - t0 + int(deltas[i]) * NS) % (L * NS)); ie_c = entry_idx(k, sh)
        ie_r = entry_idx(k, draw(rng2))
        for H in HOLDS:
            circ[H][i] = np.nanmean(fwd_avoided(k, ie_c, H)); reg[H][i] = np.nanmean(fwd_avoided(k, ie_r, H))
    NULLS[(pol, k)] = (circ, reg)
    for H in HOLDS:
        for name, nl in [('circular shift >=7d', circ[H]), (f'regime-matched (+-{REGIME_WINDOW_D}d)', reg[H])]:
            rows.append(dict(policy=pol, period=k, hold_min=H, null=name, observed_mean_avoided_bp=1e4 * obs[H],
                             null_mean_bp=1e4 * float(nl.mean()), excess_bp=1e4 * (obs[H] - float(nl.mean())),
                             null_p95_bp=1e4 * float(np.percentile(nl, 95)), p_one_sided=(1 + int((nl >= obs[H]).sum())) / (N_NULL + 1)))
TV = pd.DataFrame(rows); TV.to_csv(OUT / '3_timing_value_vs_nulls.csv', index=False)
print(f'nulls done in {time.time() - t_:.0f}s'); display(TV.round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for i, k in enumerate(PERIODS):
    if ('calendar', k) not in NULLS:
        continue
    circ, reg = NULLS[('calendar', k)]; obs = TV[(TV.policy == 'calendar') & (TV.period == k) & (TV.hold_min == PRIMARY_HOLD)].observed_mean_avoided_bp.iloc[0]
    ax[i].hist(1e4 * circ[PRIMARY_HOLD], bins=60, alpha=.6, color='#9aa7b4', label='circular shift')
    ax[i].hist(1e4 * reg[PRIMARY_HOLD], bins=60, alpha=.6, color='#d4a373', label='regime-matched')
    ax[i].axvline(obs, color='#c0392b', lw=2, label=f'actual alerts ({obs:.1f} bp)')
    ax[i].set_title(f'{k}: mean loss avoided per alert, {PRIMARY_HOLD}-min exit'); ax[i].set_xlabel('bp per alert (before fees)')
ax[0].legend(fontsize=9); plt.tight_layout(); plt.savefig(OUT / '3_timing_value.png', dpi=140); plt.show()

## Summary

In [ ]:
def pick(df, **kw):
    r = df
    for c_, v_ in kw.items():
        r = r[r[c_] == v_]
    return r.iloc[0].to_dict() if len(r) else None


summary = {'spec_sha256': SPEC_SHA}
for (pol, k) in ALERTS:
    key = f'{pol}/{k}'
    summary[key] = dict(
        post_onset_hits=pick(agg, policy=pol, period=k, endpoint='post-onset hit'),
        pre_onset_hits=pick(agg, policy=pol, period=k, endpoint='pre-onset hit'),
        coverage=COV[(COV.policy == pol) & (COV.period == k)].set_index('endpoint').share_of_all_crash_drawdown_avoided.to_dict(),
        alert_classes=AC[(AC.policy == pol) & (AC.period == k)].set_index('alert_class').alerts.to_dict(),
        net_per_year_all_alerts_pct={int(f): float(v) for f, v in NV[(NV.policy == pol) & (NV.period == k) & (NV.subset == 'all alerts')]
                                     .set_index('one_way_fee_bp').total_net_pct_per_year.items()},
        timing_value_240m={r['null']: dict(excess_bp=r['excess_bp'], p=r['p_one_sided'])
                           for r in TV[(TV.policy == pol) & (TV.period == k) & (TV.hold_min == PRIMARY_HOLD)].to_dict('records')})
(OUT / 'VALUE_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print(json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))[:6000])